In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT_DIR = Path("..").resolve()
DATA_DIR = ROOT_DIR / "data"

TRAIN_TX_PATH = DATA_DIR / "atm_transactions_train.csv"
TEST_TX_PATH  = DATA_DIR / "atm_transactions_test.csv"  # or your actual test file name
CAL_PATH      = DATA_DIR / "calendar.csv"
META_PATH     = DATA_DIR / "atm_metadata.csv"
REGION_PATH   = DATA_DIR / "atm_region_lookup.csv"
REPL_PATH     = DATA_DIR / "cash_replenishment.csv"

print("Train exists:", TRAIN_TX_PATH.exists())
print("Test exists:", TEST_TX_PATH.exists())
print("Calendar exists:", CAL_PATH.exists())
print("Metadata exists:", META_PATH.exists())
print("Region lookup exists:", REGION_PATH.exists())
print("Replenishment exists:", REPL_PATH.exists())

Train exists: True
Test exists: True
Calendar exists: True
Metadata exists: True
Region lookup exists: True
Replenishment exists: True


In [5]:
def load_clean_transactions(train_path: Path, test_path: Path):
    # --- Train ---
    train = pd.read_csv(train_path)

    train = train.rename(columns={
        "dt": "dt",
        "atm_id": "atm_id",
        "total_withdrawn_amount_kwd": "withdrawn_kwd",
        "total_withdraw_txn_count": "withdraw_count",
    })

    train["dt"] = pd.to_datetime(train["dt"])
    # Aggregate per ATM-day just in case
    train = (
        train[["dt", "atm_id", "withdrawn_kwd", "withdraw_count"]]
        .drop_duplicates()
        .groupby(["atm_id", "dt"], as_index=False)[["withdrawn_kwd", "withdraw_count"]]
        .sum()
    )
    train["is_train"] = 1

    # --- Test ---
    test = pd.read_csv(test_path)
    test["dt"] = pd.to_datetime(test["dt"])

    test = test[["dt", "atm_id"]].copy()
    test["withdrawn_kwd"] = np.nan
    test["withdraw_count"] = np.nan
    test["is_train"] = 0

    return train, test


df_train_tx, df_test_tx = load_clean_transactions(TRAIN_TX_PATH, TEST_TX_PATH)
df_train_tx.head(), df_test_tx.head()

(     atm_id         dt  withdrawn_kwd  withdraw_count  is_train
 0  ATM_0001 2020-07-16         713.23              31         1
 1  ATM_0001 2020-07-17         955.08              33         1
 2  ATM_0001 2020-07-18         974.74              33         1
 3  ATM_0001 2020-07-19         688.74              24         1
 4  ATM_0001 2020-07-20        1360.88              47         1,
           dt    atm_id  withdrawn_kwd  withdraw_count  is_train
 0 2025-10-28  ATM_0004            NaN             NaN         0
 1 2025-10-28  ATM_0005            NaN             NaN         0
 2 2025-10-28  ATM_0006            NaN             NaN         0
 3 2025-10-28  ATM_0007            NaN             NaN         0
 4 2025-10-28  ATM_0009            NaN             NaN         0)

In [6]:
def load_calendar(path: Path) -> pd.DataFrame:
    cal = pd.read_csv(path)
    cal["dt"] = pd.to_datetime(cal["dt"])

    # Ensure boolean-ish columns are numeric 0/1
    for col in ["is_weekend", "is_public_holiday",
                "is_salary_disbursement", "is_ramadan"]:
        if col in cal.columns:
            cal[col] = cal[col].astype(int)

    # We can keep holiday_name as a categorical feature later if we want
    return cal


cal = load_calendar(CAL_PATH)
cal.head()

,dt,is_weekend,is_public_holiday,holiday_name,is_salary_disbursement,is_ramadan,days_to_salary,days_from_salary,week_of_year,month,quarter,year
0,2020-01-01,0,1,New Year’s Day,0,0,24,7,1,1,1,2020
1,2020-01-02,0,0,NaN,0,0,23,8,1,1,1,2020
2,2020-01-03,1,0,NaN,0,0,22,9,1,1,1,2020
3,2020-01-04,1,0,NaN,0,0,21,10,1,1,1,2020
4,2020-01-05,0,0,NaN,0,0,20,11,1,1,1,2020


In [7]:
def load_atm_metadata(meta_path: Path, region_path: Path) -> pd.DataFrame:
    meta = pd.read_csv(meta_path)

    # Rename to explicit meta names
    meta = meta.rename(columns={
        "region": "atm_region_meta",
        "location_type": "location_type_meta",
    })

    # Parse dates
    if "installed_date" in meta.columns:
        meta["installed_date"] = pd.to_datetime(meta["installed_date"])
    if "decommissioned_date" in meta.columns:
        meta["decommissioned_date"] = pd.to_datetime(meta["decommissioned_date"])

    # Region lookup
    reg = pd.read_csv(region_path)
    reg = reg.rename(columns={
        "region": "region_lookup",
        "location_type": "location_type_lookup",
    })

    meta = meta.merge(reg, on="atm_id", how="left")

    # Region mismatch flag
    if "atm_region_meta" in meta.columns and "region_lookup" in meta.columns:
        meta["region_mismatch_flag"] = (
            meta["atm_region_meta"] != meta["region_lookup"]
        ).astype(int)

    return meta


meta = load_atm_metadata(META_PATH, REGION_PATH)
meta.head()

,atm_id,name,location_type_meta,atm_region_meta,latitude,longitude,installed_date,decommissioned_date,region_lookup,location_type_lookup,region_mismatch_flag
0,ATM_0001,Hawalli Branch 1,branch,Hawalli,29.129733,48.127402,2020-07-16,2024-01-14,Hawalli,branch,0
1,ATM_0002,Kuwait City Branch 2,branch,Kuwait City,29.161132,47.998213,2022-09-12,2024-05-18,Kuwait City,branch,0
2,ATM_0003,Farwaniya Branch 3,branch,Farwaniya,29.239387,47.836218,2022-06-24,2023-09-19,Farwaniya,branch,0
3,ATM_0004,Ahmadi Branch 4,branch,Ahmadi,29.090487,47.965822,2022-02-08,NaT,Ahmadi,branch,0
4,ATM_0005,Ahmadi Branch 5,branch,Ahmadi,29.216542,48.016860,2023-07-10,NaT,Ahmadi,branch,0


In [8]:
def load_replenishment(path: Path) -> pd.DataFrame:
    repl = pd.read_csv(path)
    repl["dt"] = pd.to_datetime(repl["dt"])

    # Daily aggregation per ATM-day
    daily = repl.groupby(["atm_id", "dt"], as_index=False).agg({
        "starting_cash_kwd": "first",
        "withdrawn_kwd": "sum",
        "deposited_kwd": "sum",
        "replenished_kwd": "sum",
        "ending_cash_kwd": "last",
        "cashout_flag": "max",
    })

    daily = daily.rename(columns={
        "starting_cash_kwd": "repl_starting_cash_kwd",
        "withdrawn_kwd": "repl_withdrawn_kwd",
        "deposited_kwd": "repl_deposited_kwd",
        "replenished_kwd": "repl_replenished_kwd",
        "ending_cash_kwd": "repl_ending_cash_kwd",
        "cashout_flag": "repl_cashout_flag",
    })

    # Ensure flag is 0/1
    daily["repl_cashout_flag"] = daily["repl_cashout_flag"].astype(int)

    return daily


repl_daily = load_replenishment(REPL_PATH)
repl_daily.head()

,atm_id,dt,repl_starting_cash_kwd,repl_withdrawn_kwd,repl_deposited_kwd,repl_replenished_kwd,repl_ending_cash_kwd,repl_cashout_flag
0,ATM_0001,2020-07-16,59383.43,713.23,22.05,0.0,58233.56,0
1,ATM_0001,2020-07-17,58233.56,955.08,10.00,0.0,56494.56,0
2,ATM_0001,2020-07-18,56494.56,974.74,15.38,0.0,56162.00,0
3,ATM_0001,2020-07-19,56162.00,688.74,0.00,0.0,55419.55,0
4,ATM_0001,2020-07-20,55419.55,1360.88,34.18,0.0,54170.16,0


In [19]:
# ------------------------------------------------------------------
# Metric
# ------------------------------------------------------------------
def rmse(y_true, y_pred) -> float:
    """
    Compute RMSE without relying on sklearn version details.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

In [9]:
# Combine train + test
df_all = pd.concat([df_train_tx, df_test_tx], ignore_index=True)
df_all = df_all.sort_values(["atm_id", "dt"]).reset_index(drop=True)

print(df_all.shape)
df_all.head()

(240222, 5)


,atm_id,dt,withdrawn_kwd,withdraw_count,is_train
0,ATM_0001,2020-07-16,713.23,31.0,1
1,ATM_0001,2020-07-17,955.08,33.0,1
2,ATM_0001,2020-07-18,974.74,33.0,1
3,ATM_0001,2020-07-19,688.74,24.0,1
4,ATM_0001,2020-07-20,1360.88,47.0,1


In [10]:
# Join calendar on dt
df_all = df_all.merge(cal, on="dt", how="left")

# Join metadata on atm_id
df_all = df_all.merge(meta, on="atm_id", how="left")

# Join replenishment per atm_id, dt
df_all = df_all.merge(repl_daily, on=["atm_id", "dt"], how="left")

In [11]:
if "installed_date" in df_all.columns:
    df_all["atm_age_days"] = (df_all["dt"] - df_all["installed_date"]).dt.days
    df_all["is_new_atm"] = (df_all["atm_age_days"] < 60).astype(int)
else:
    df_all["atm_age_days"] = np.nan
    df_all["is_new_atm"] = 0

In [12]:
df_all = df_all.sort_values(["atm_id", "dt"]).reset_index(drop=True)

# Indicator for a replenishment on that day (any positive replenished amount)
df_all["had_repl_today"] = (
    df_all["repl_replenished_kwd"].fillna(0) > 0
)

# Last replenishment date per ATM
df_all["last_repl_dt"] = (
    df_all["dt"]
    .where(df_all["had_repl_today"])
    .groupby(df_all["atm_id"])
    .ffill()
)

df_all["days_since_last_repl"] = (
    df_all["dt"] - df_all["last_repl_dt"]
).dt.days

# Fill NaNs (no replenishment yet) with a large number
df_all["days_since_last_repl"] = df_all["days_since_last_repl"].fillna(999)

# Clean up helpers
df_all.drop(columns=["had_repl_today", "last_repl_dt"], inplace=True)

In [13]:
def add_lag_features(df: pd.DataFrame, lags=(1, 7, 14, 28)) -> pd.DataFrame:
    df = df.sort_values(["atm_id", "dt"]).copy()
    for lag in lags:
        df[f"lag{lag}_amt"] = df.groupby("atm_id")["withdrawn_kwd"].shift(lag)
        df[f"lag{lag}_cnt"] = df.groupby("atm_id")["withdraw_count"].shift(lag)
    return df


def add_rolling_features(df: pd.DataFrame, windows=(7, 28)) -> pd.DataFrame:
    df = df.sort_values(["atm_id", "dt"]).copy()
    for w in windows:
        df[f"roll{w}_amt_mean"] = (
            df.groupby("atm_id")["withdrawn_kwd"]
            .shift(1)   # ensure only past info
            .rolling(window=w, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
        )
        df[f"roll{w}_cnt_mean"] = (
            df.groupby("atm_id")["withdraw_count"]
            .shift(1)
            .rolling(window=w, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
        )
    return df


df_all = add_lag_features(df_all)
df_all = add_rolling_features(df_all)

print(df_all.shape)
df_all.head()

(240222, 47)


,atm_id,dt,withdrawn_kwd,withdraw_count,is_train,is_weekend,is_public_holiday,holiday_name,is_salary_disbursement,is_ramadan,...,lag7_amt,lag7_cnt,lag14_amt,lag14_cnt,lag28_amt,lag28_cnt,roll7_amt_mean,roll7_cnt_mean,roll28_amt_mean,roll28_cnt_mean
0,ATM_0001,2020-07-16,713.23,31.0,1,0,0,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ATM_0001,2020-07-17,955.08,33.0,1,1,0,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,713.230000,31.000000,713.230000,31.000000
2,ATM_0001,2020-07-18,974.74,33.0,1,1,0,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,834.155000,32.000000,834.155000,32.000000
3,ATM_0001,2020-07-19,688.74,24.0,1,0,0,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,881.016667,32.333333,881.016667,32.333333
4,ATM_0001,2020-07-20,1360.88,47.0,1,0,0,NaN,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,832.947500,30.250000,832.947500,30.250000


In [14]:
import numpy as np

# Use only training rows (where we have labels)
df_train_all = df_all[df_all["is_train"] == 1].copy()
df_test_all  = df_all[df_all["is_train"] == 0].copy()

# Time-based split: last 14 days as validation
max_dt = df_train_all["dt"].max()
val_start = max_dt - pd.Timedelta(days=13)

df_tr = df_train_all[df_train_all["dt"] < val_start].copy()
df_val = df_train_all[df_train_all["dt"] >= val_start].copy()

print("Train date range:", df_tr["dt"].min().date(), "->", df_tr["dt"].max().date())
print("Val date range:  ", df_val["dt"].min().date(), "->", df_val["dt"].max().date())
print("#Train rows:", len(df_tr), "#Val rows:", len(df_val))

Train date range: 2020-01-04 -> 2025-10-13
Val date range:   2025-10-14 -> 2025-10-27
#Train rows: 234461 #Val rows: 2853


In [15]:
# Columns we definitely don't want as features
drop_cols = [
    "withdrawn_kwd", "withdraw_count",  # targets
    "is_train",
    "dt", "atm_id",
    "installed_date", "decommissioned_date",
    "name", "holiday_name",  # free-text columns
]

feature_cols = [c for c in df_tr.columns if c not in drop_cols]

print("Number of feature columns:", len(feature_cols))

# Identify categorical vs numeric
cat_cols = [c for c in feature_cols if df_tr[c].dtype == "object"]
num_cols = [c for c in feature_cols if c not in cat_cols]

print("Categorical cols:", cat_cols)
print("Numeric cols:", len(num_cols))

Number of feature columns: 38
Categorical cols: ['location_type_meta', 'atm_region_meta', 'region_lookup', 'location_type_lookup']
Numeric cols: 34


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

In [17]:
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

In [20]:
# Model 1 – Predict withdrawal amount (withdrawn_kwd) with log-transform
# Targets
y_tr_amt  = np.log1p(df_tr["withdrawn_kwd"].values)
y_val_amt = np.log1p(df_val["withdrawn_kwd"].values)

X_tr = df_tr[feature_cols]
X_val = df_val[feature_cols]

model_amt = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("xgb", XGBRegressor(
            n_estimators=400,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            n_jobs=4,
            tree_method="hist",  # ok to keep; if error, just remove this argument
        )),
    ]
)

print("Training XGB model for withdrawal amount...")
model_amt.fit(X_tr, y_tr_amt)

# Predict on validation, invert log transform
val_pred_log = model_amt.predict(X_val)
val_pred_amt = np.expm1(val_pred_log)

rmse_kwd_ml = rmse(df_val["withdrawn_kwd"], val_pred_amt)
print("Validation RMSE (withdrawn_kwd) - ML model:", rmse_kwd_ml)

Training XGB model for withdrawal amount...
Validation RMSE (withdrawn_kwd) - ML model: 53.75168911032135


In [21]:
# Model 2 – Predict withdrawal count (withdraw_count)
y_tr_cnt  = np.log1p(df_tr["withdraw_count"].values)
y_val_cnt = np.log1p(df_val["withdraw_count"].values)

model_cnt = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("xgb", XGBRegressor(
            n_estimators=400,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            n_jobs=4,
            tree_method="hist",
        )),
    ]
)

print("Training XGB model for withdrawal count...")
model_cnt.fit(X_tr, y_tr_cnt)

val_pred_log_cnt = model_cnt.predict(X_val)
val_pred_cnt = np.expm1(val_pred_log_cnt)

rmse_cnt_ml = rmse(df_val["withdraw_count"], val_pred_cnt)
print("Validation RMSE (withdraw_count) - ML model:", rmse_cnt_ml)

avg_rmse_ml = (rmse_kwd_ml + rmse_cnt_ml) / 2.0
print("Average RMSE - ML models:", avg_rmse_ml)

Training XGB model for withdrawal count...
Validation RMSE (withdraw_count) - ML model: 3.5415385975412805
Average RMSE - ML models: 28.646613853931317
